In [310]:
import polars as pl
from pathlib import Path
import altair as alt

# Pretraining Data


In [311]:
MAKE_PRETRAIN_CSV = True
latest_pretrain_log = max(
    Path("logs").glob("pretrain_*.log"), key=lambda x: x.stat().st_mtime
)
pretrain_log_file = latest_pretrain_log
pretrain_csv = Path("pretrain_log.csv")

if MAKE_PRETRAIN_CSV:
    with open(pretrain_log_file, "r") as f:
        lines = [line.split("INFO")[1].strip() for line in f if "step" in line]
        data = []
        for line in lines:
            parts = line.split("|")
            step = int(parts[0].split()[1])
            loss = float(parts[1].split()[1])
            acc = float(parts[2].split()[1][:-1])
            episodes = int(parts[3].split()[1])
            time = float(parts[4].split()[0][:-1])
            data.append((step, loss, acc, episodes, time))
        pretrain_df = pl.DataFrame(
            data, schema=["step", "loss", "acc", "episodes", "time"], orient="row"
        )
    pretrain_df.write_csv(pretrain_csv)

# PPO Training Data


In [312]:
MAKE_PPO_CSV = True
latest_ppo_log = max(Path("logs").glob("ppo_*.log"), key=lambda x: x.stat().st_mtime)
ppo_log_file = latest_ppo_log
ppo_warmup_csv = Path("ppo_warmup_log.csv")
ppo_csv = Path("ppo_log.csv")
bayesian_baseline_turns = 100  # Max turns

if MAKE_PPO_CSV:
    with open(ppo_log_file, "r") as f:
        warmup_line = []
        lines = []
        for line in f:
            if "warmup" in line and "value" in line:
                warmup_line.append(line.split("INFO")[1].strip())
            elif "entropy" in line:
                lines.append(line.split("INFO")[1].strip())
            elif "Baseline" in line:
                bayesian_baseline_turns = float(
                    line.split("Baseline")[1].strip().split(" ")[1]
                )

        warmup_data = []
        for line in warmup_line:
            parts = line.split("|")
            it = int(parts[0].split()[1].split("/")[0])
            value = float(parts[1].split()[1])
            lr = float(parts[2].split()[1])
            warmup_data.append((it, value, lr))
        data = []
        for line in lines:
            parts = line.split("|")
            it = int(parts[0].split()[1])
            turns = float(parts[1].split()[1])
            policy = float(parts[2].split()[1])
            value = float(parts[3].split()[1])
            entropy = float(parts[4].split()[1])
            time = float(parts[5].split()[0][:-1])
            data.append((it, turns, policy, value, entropy, time))

    ppo_warmup_df = pl.DataFrame(
        warmup_data,
        schema=["iter", "value", "lr"],
        orient="row",
    )
    ppo_warmup_df.write_csv(ppo_warmup_csv)
    ppo_df = pl.DataFrame(
        data,
        schema=["iter", "turns", "policy", "value", "entropy", "time"],
        orient="row",
    )
    ppo_df.write_csv(ppo_csv)

In [313]:
# How big the rolling window should be for the charts
WINDOW_SIZE = 10


def make_chart(
    df: pl.DataFrame,
    y_col: str,
    f,
    window_size: int = WINDOW_SIZE,
    arg_col: str = "step",
    scale_x: bool = True,
) -> alt.Chart | alt.LayerChart | alt.FacetChart:
    df = df.with_columns(
        pl.col(y_col).rolling_mean(window_size=window_size).alias(f"{y_col}_rolling"),
    )
    # Chart size calculations
    x_max = df[arg_col].max() * 1.05
    x_axis = alt.Axis()
    if scale_x:
        x_axis = alt.Axis(labelExpr="datum.value / 1000 + 'k'")
    x_scale = alt.Scale(domainMax=x_max)

    # Define color scale for lines and rules
    color_domain = ["Rolling avg", "Best rolling", "Best raw"]
    color_range = ["steelblue", "grey", "red"]
    color_scale = alt.Scale(domain=color_domain, range=color_range)
    # Rolling average line
    line = (
        alt.Chart(df.with_columns(pl.lit("Rolling avg").alias("series")))
        .mark_line()
        .encode(
            x=alt.X(arg_col, title=arg_col.capitalize(), axis=x_axis, scale=x_scale),
            y=alt.Y(
                f"{y_col}_rolling",
                title=f"{y_col.capitalize()} ({window_size}-{arg_col} rolling avg)",
            ),
            color=alt.Color("series:N", scale=color_scale, legend=alt.Legend(title="")),
        )
        .properties(
            title=f"{y_col.capitalize()} v {arg_col.capitalize()}",
            width=500,
            height=300,
        )
    )

    if not f:
        return line

    # Best rolling and raw value rules
    h_rolling_val = df.select(f(f"{y_col}_rolling")).item()
    rule_rolling = (
        alt.Chart(pl.DataFrame({"y": [h_rolling_val], "series": ["Best rolling"]}))
        .mark_rule(fillOpacity=0.8, strokeDash=[4, 4])
        .encode(
            y=alt.Y("y:Q"),
            color=alt.Color("series:N", scale=color_scale, legend=alt.Legend(title="")),
        )
    )
    h_val = df.select(f(y_col)).item()
    rule = (
        alt.Chart(pl.DataFrame({"y": [h_val], "series": ["Best raw"]}))
        .mark_rule(fillOpacity=0.8, strokeDash=[4, 4])
        .encode(
            y=alt.Y("y:Q"),
            color=alt.Color("series:N", scale=color_scale, legend=alt.Legend(title="")),
        )
    )

    return line + rule_rolling + rule

---

# Imitation Pretraining Analysis


In [314]:
pretrain_df = pl.read_csv(pretrain_csv)
pretrain_df.show()

step,loss,acc,episodes,time
i64,f64,f64,i64,f64
1,4.5838,0.0,0,0.0
1000,1.8563,52.4,21,15.2
2000,2.214,47.1,43,16.0
3000,4.3377,1.5,64,15.5
4000,4.2941,3.6,86,15.5


In [315]:
loss = make_chart(pretrain_df, "loss", pl.min)
acc = make_chart(pretrain_df, "acc", pl.max)

# Total time
total_seconds = pretrain_df["time"].sum()
h, m = divmod(total_seconds, 3600)
m, s = divmod(m, 60)
print(f"Total training time: {int(h)}h {int(m)}m {int(s)}s")
# Min loss and max accuracy
min_loss = pretrain_df["loss"].min()
max_acc = pretrain_df["acc"].max()
print(f"Min loss: {min_loss:.4f} | Max accuracy: {max_acc:.1f}%")
# Display charts
title = alt.TitleParams(
    "Transformer PPO - Imitation Pretraining Metrics", anchor="middle"
)
(loss | acc).resolve_scale(y="independent").properties(title=title)

Total training time: 0h 27m 43s
Min loss: 0.4933 | Max accuracy: 90.2%


alt.HConcatChart(...)

---

# PPO Analysis


In [316]:
ppo_warmup_df = pl.read_csv(ppo_warmup_csv)
ppo_warmup_df.show()

iter,value,lr
i64,f64,f64
5,113.0741,0.000298
10,85.5626,0.000293
15,60.4693,0.000284
20,44.125,0.000272
25,22.8848,0.000257


In [317]:
value, lr = [
    make_chart(ppo_warmup_df, col, f, arg_col="iter", scale_x=False, window_size=1)
    for col, f in [
        ("value", pl.min),
        ("lr", None),
    ]
]

# Display charts
title = alt.TitleParams("Transformer PPO - Warmup Metrics", anchor="middle")
((value | lr)).resolve_scale(y="independent").properties(title=title)

alt.HConcatChart(...)

In [318]:
ppo_df = pl.read_csv(ppo_csv)
ppo_df.show()

iter,turns,policy,value,entropy,time
i64,f64,f64,f64,f64,f64
1,50.7,0.0555,0.7135,0.0783,112.49
2,45.8,0.1046,0.9933,0.0824,101.0
3,43.1,0.0898,0.9668,0.09,100.06
4,48.6,0.1321,0.8324,0.1129,116.91
5,47.6,0.0586,0.9581,0.0969,120.03


In [ ]:
turns, policy, value, entropy = [
    make_chart(ppo_df, col, f, arg_col="iter", scale_x=False, window_size=5)
    for col, f in [
        ("turns", pl.min),
        ("policy", pl.min),
        ("value", pl.min),
        ("entropy", pl.min),
    ]
]


bayes_target = bayesian_baseline_turns
rule = (
    alt.Chart(pl.DataFrame({"y": [bayes_target], "series": ["Bayes target"]}))
    .mark_rule(strokeDash=[4, 2])
    .encode(
        y=alt.Y("y:Q"),
        color=alt.Color(
            "series:N",
            scale=alt.Scale(domain=["Bayes target"], range=["black"]),
            legend=alt.Legend(title=""),
        ),
    )
)
turns = turns + rule

# Total time
total_seconds = ppo_df["time"].sum()
h, m = divmod(total_seconds, 3600)
m, s = divmod(m, 60)
print(f"Total training time: {int(h)}h {int(m)}m {int(s)}s")
# Display charts
title = alt.TitleParams("Transformer PPO - Training Metrics", anchor="middle")
((turns | value) & (policy | entropy)).resolve_scale(y="independent").properties(
    title=title
)

Total training time: 2h 38m 50s


alt.VConcatChart(...)